# CINF104 - Proyecto 1: Predicción GRD Hospital El Pino
## 02 - Preprocesamiento y Feature Engineering

**Decisión de target:** `grd_full` con estrategia **long-tail** (umbral = 10 ejemplos mínimos)  
- Clases frecuentes: 229 GRDs distintos  
- Clase `OTROS`: agrupa GRDs con < 10 ejemplos (~7.6% de pacientes)  
- Total clases: 230  

**Rationale:** Con umbral 10 conservamos el 92.4% de los datos con etiqueta específica y mantenemos un número manejable de clases.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import re
import joblib

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)

DATA    = Path('../data/dataset_elpino.csv')
OUT_DIR = Path('../data/processed')
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED     = 42
MIN_EJEMPLOS    = 10   # umbral long-tail
CLASS_OTROS     = 'OTROS'

df = pd.read_csv(DATA, sep=None, engine='python')
print(f'Dataset cargado: {len(df):,} filas × {len(df.columns)} columnas')

Dataset cargado: 14,561 filas × 68 columnas


## 1. Preparación de la variable objetivo

In [2]:
# Extraer código GRD limpio
def extraer_codigo_grd(texto):
    if pd.isna(texto):
        return None
    return texto.split(' - ')[0].strip()

df['grd_full'] = df['GRD'].apply(extraer_codigo_grd)

# Aplicar estrategia long-tail
conteos = df['grd_full'].value_counts()
clases_validas = set(conteos[conteos >= MIN_EJEMPLOS].index)

df['target'] = df['grd_full'].apply(
    lambda x: x if x in clases_validas else CLASS_OTROS
)

print(f'GRDs únicos originales: {df["grd_full"].nunique()}')
print(f'Clases en target (umbral={MIN_EJEMPLOS}): {df["target"].nunique()}')
print(f'Pacientes en OTROS: {(df["target"] == CLASS_OTROS).sum():,} ({(df["target"] == CLASS_OTROS).mean()*100:.1f}%)')

GRDs únicos originales: 526
Clases en target (umbral=10): 230
Pacientes en OTROS: 1,107 (7.6%)


## 2. Identificar columnas por tipo

In [3]:
diag_cols   = [c for c in df.columns if 'Diag' in c]
proced_cols = [c for c in df.columns if 'Proced' in c]

print(f'Columnas de diagnóstico: {len(diag_cols)}')
print(f'Columnas de procedimiento: {len(proced_cols)}')
print(f'Diag cols: {diag_cols[:3]} ...')
print(f'Proced cols: {proced_cols[:3]} ...')

Columnas de diagnóstico: 35
Columnas de procedimiento: 30
Diag cols: ['Diag 01 Principal (cod+des)', 'Diag 02 Secundario (cod+des)', 'Diag 03 Secundario (cod+des)'] ...
Proced cols: ['Proced 01 Principal (cod+des)', 'Proced 02 Secundario (cod+des)', 'Proced 03 Secundario (cod+des)'] ...


## 3. Extracción de códigos CIE desde texto libre

In [4]:
def extraer_codigo(texto):
    """Extrae el código CIE del formato 'A41.8 - Descripción'."""
    if pd.isna(texto):
        return None
    codigo = texto.split(' - ')[0].strip()
    # Normalizar: quitar punto y tomar prefijo de 3 caracteres (nivel 3-char CIE)
    return codigo[:3] if len(codigo) >= 3 else codigo

# Extraer códigos para diagnósticos y procedimientos
for col in diag_cols:
    df[col + '_cod'] = df[col].apply(extraer_codigo)

for col in proced_cols:
    df[col + '_cod'] = df[col].apply(extraer_codigo)

diag_cod_cols   = [c + '_cod' for c in diag_cols]
proced_cod_cols = [c + '_cod' for c in proced_cols]

print('Ejemplo diagnóstico principal extraído:')
print(df[['Diag 01 Principal (cod+des)', 'Diag 01 Principal (cod+des)_cod']].head(5))

Ejemplo diagnóstico principal extraído:


                                   Diag 01 Principal (cod+des)  \
0                      A41.8 - Otras septicemias especificadas   
1                         U07.1 - COVID-19, virus identificado   
2    K56.5 - Adherencias [bridas] intestinales con obstrucción   
3          K76.8 - Otras enfermedades especificadas del hígado   
4  T81.0 - Hemorragia y hematoma que complican un procedimi...   

  Diag 01 Principal (cod+des)_cod  
0                             A41  
1                             U07  
2                             K56  
3                             K76  
4                             T81  


## 4. Features numéricas base

In [5]:
# Contar diagnósticos y procedimientos por paciente
df['n_diagnosticos']   = df[diag_cod_cols].notna().sum(axis=1)
df['n_procedimientos'] = df[proced_cod_cols].notna().sum(axis=1)

# Edad (ya numérica)
df['edad'] = pd.to_numeric(df['Edad en años'], errors='coerce')
df['edad'] = df['edad'].fillna(df['edad'].median())

# Sexo → binario
df['sexo_bin'] = (df['Sexo (Desc)'].str.upper().str.strip() == 'MASCULINO').astype(int)

print('Estadísticas de features numéricas:')
print(df[['edad','n_diagnosticos','n_procedimientos','sexo_bin']].describe())

Estadísticas de features numéricas:
               edad  n_diagnosticos  n_procedimientos  sexo_bin
count  14561.000000         14561.0           14561.0   14561.0
mean      39.426550            35.0              30.0       0.0
std       24.681545             0.0               0.0       0.0
min        0.000000            35.0              30.0       0.0
25%       23.000000            35.0              30.0       0.0
50%       36.000000            35.0              30.0       0.0
75%       60.000000            35.0              30.0       0.0
max      121.000000            35.0              30.0       0.0


## 5. Multi-hot encoding de diagnósticos y procedimientos

Estrategia: tomamos los **N** códigos CIE más frecuentes del vocabulario de diagnósticos y procedimientos, y creamos un vector binario por fila.

In [6]:
from sklearn.preprocessing import MultiLabelBinarizer

TOP_DIAG_VOCAB   = 200   # top códigos de diagnóstico
TOP_PROCED_VOCAB = 100   # top códigos de procedimiento

def build_multihot(df, cod_cols, top_n, vocab=None):
    """
    Construye un multi-hot encoding para columnas de códigos.
    Retorna la matriz binaria y el vocabulario usado.
    """
    # Construir listas de códigos por fila (ignorar None)
    listas = df[cod_cols].apply(
        lambda row: [c for c in row if pd.notna(c)], axis=1
    )
    
    if vocab is None:
        # Calcular frecuencia de cada código
        from collections import Counter
        freq = Counter(c for lista in listas for c in lista)
        vocab = [c for c, _ in freq.most_common(top_n)]
    
    # Filtrar listas al vocabulario
    vocab_set = set(vocab)
    listas_filtradas = listas.apply(lambda lst: [c for c in lst if c in vocab_set])
    
    mlb = MultiLabelBinarizer(classes=vocab)
    matriz = mlb.fit_transform(listas_filtradas)
    cols = [f'diag_{v}' if 'diag' in cod_cols[0].lower() else f'proc_{v}' for v in vocab]
    return pd.DataFrame(matriz, columns=cols, index=df.index), vocab

# Diagnósticos
df_diag_mh, vocab_diag = build_multihot(df, diag_cod_cols, TOP_DIAG_VOCAB)
print(f'Multi-hot diagnósticos: {df_diag_mh.shape}')

# Procedimientos
df_proc_mh, vocab_proc = build_multihot(df, proced_cod_cols, TOP_PROCED_VOCAB)
print(f'Multi-hot procedimientos: {df_proc_mh.shape}')

Multi-hot diagnósticos: (14561, 200)


Multi-hot procedimientos: (14561, 83)


## 6. Ensamblado del feature set final

In [7]:
# Features numéricas base
features_base = df[['edad', 'n_diagnosticos', 'n_procedimientos', 'sexo_bin']].copy()

# Concatenar todo
X = pd.concat([features_base.reset_index(drop=True),
               df_diag_mh.reset_index(drop=True),
               df_proc_mh.reset_index(drop=True)], axis=1)

y = df['target'].values

print(f'Shape del dataset final: X={X.shape}, y={y.shape}')
print(f'Número de clases: {len(np.unique(y))}')
print(f'Features: {X.columns.tolist()[:10]} ...')

Shape del dataset final: X=(14561, 287), y=(14561,)
Número de clases: 230
Features: ['edad', 'n_diagnosticos', 'n_procedimientos', 'sexo_bin', 'diag_-', 'diag_Z39', 'diag_Z92', 'diag_I10', 'diag_Z38', 'diag_Z37'] ...


## 7. Label encoding del target

In [8]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_enc = le.fit_transform(y)

print(f'Clases codificadas: {len(le.classes_)}')
print(f'Ejemplo: {le.classes_[:5]} → {le.transform(le.classes_[:5])}')

Clases codificadas: 230
Ejemplo: ['014132' '014133' '014141' '014142' '014143'] → [0 1 2 3 4]


## 8. Split estratificado train / val / test

In [9]:
from sklearn.model_selection import train_test_split

# 70% train | 15% val | 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_enc,
    test_size=0.30,
    random_state=RANDOM_SEED,
    stratify=y_enc
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=RANDOM_SEED,
    stratify=y_temp
)

print(f'Train: {X_train.shape[0]:,} | Val: {X_val.shape[0]:,} | Test: {X_test.shape[0]:,}')

Train: 10,192 | Val: 2,184 | Test: 2,185


## 9. Normalización de features continuas

In [10]:
from sklearn.preprocessing import StandardScaler

cont_cols = ['edad', 'n_diagnosticos', 'n_procedimientos']
cont_idx  = [X.columns.tolist().index(c) for c in cont_cols]

scaler = StandardScaler()
X_train_arr = X_train.values.astype(np.float32)
X_val_arr   = X_val.values.astype(np.float32)
X_test_arr  = X_test.values.astype(np.float32)

X_train_arr[:, cont_idx] = scaler.fit_transform(X_train_arr[:, cont_idx])
X_val_arr[:, cont_idx]   = scaler.transform(X_val_arr[:, cont_idx])
X_test_arr[:, cont_idx]  = scaler.transform(X_test_arr[:, cont_idx])

print(f'Normalización aplicada en columnas: {cont_cols}')
print(f'Shape final arrays: train={X_train_arr.shape}')

Normalización aplicada en columnas: ['edad', 'n_diagnosticos', 'n_procedimientos']
Shape final arrays: train=(10192, 287)


## 10. Guardar datasets procesados y artefactos

In [11]:
import joblib

# Guardar arrays numpy
np.save(OUT_DIR / 'X_train.npy', X_train_arr)
np.save(OUT_DIR / 'X_val.npy',   X_val_arr)
np.save(OUT_DIR / 'X_test.npy',  X_test_arr)
np.save(OUT_DIR / 'y_train.npy', y_train)
np.save(OUT_DIR / 'y_val.npy',   y_val)
np.save(OUT_DIR / 'y_test.npy',  y_test)

# Guardar transformadores y vocabulario
joblib.dump(le,     OUT_DIR / 'label_encoder.pkl')
joblib.dump(scaler, OUT_DIR / 'scaler.pkl')
joblib.dump({'diag': vocab_diag, 'proc': vocab_proc}, OUT_DIR / 'vocabulario.pkl')

# Guardar metadata
meta = {
    'n_train': len(y_train),
    'n_val':   len(y_val),
    'n_test':  len(y_test),
    'n_features': X_train_arr.shape[1],
    'n_clases': len(le.classes_),
    'umbral_long_tail': MIN_EJEMPLOS,
    'vocab_diag': len(vocab_diag),
    'vocab_proc': len(vocab_proc),
}
import json
with open(OUT_DIR / 'metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('✅ Datasets y artefactos guardados en:', OUT_DIR)
print(json.dumps(meta, indent=2))

✅ Datasets y artefactos guardados en: ../data/processed
{
  "n_train": 10192,
  "n_val": 2184,
  "n_test": 2185,
  "n_features": 287,
  "n_clases": 230,
  "umbral_long_tail": 10,
  "vocab_diag": 200,
  "vocab_proc": 83
}
